### Similarity Module


In [ ]:
import sys
import hydra
import torch
from omegaconf import OmegaConf
from pathlib import Path

In [ ]:
def SimilarityModule(
    query_embedding: torch.Tensor,
    support_set_embeddings: torch.Tensor,
    padding_mask: torch.Tensor,
    support_set_size: torch.Tensor,
    cfg: OmegaConf,
) -> torch.Tensor:
    """
    The similarity module builds the activity prediction for the query molecule from a
    weighted sum over the support set labels. Pair-wise similarity values between query
    and support set molecules are used as weights for the weighted sum.

    Since the similarity module is applied twice within the MHNfs model - once for the
    active and once for the inactive support set molecules, the support_set_embeddings
    here mean ether active or inactive support set molecule embeddings.

    inputs:
    - query; torch.tensor;
      dim: [batch-size, 1, embedding-dimension]
        * e.g.: [512, 1, 1024]
    - support set molecules; torch.tensor;
      dim: [batch-size, padding-dim, embedding-dimension]
        * e.g.: [512, 9, 1024]
    - padding mask; torch.tensor; boolean
      dim: [batch-size, padding-dim]
        * e.g.: [512, 9]
    - support set size; torch.tensor;
      dim: [batch-size]
        * e.g.: [512]
    """

    # Optional L2-norm
    if cfg.model.similarityModule.l2Norm:
        query_embedding_div = torch.unsqueeze(
            query_embedding.pow(2).sum(dim=2).sqrt(), 2
        )
        query_embedding_div[query_embedding_div == 0] = 1
        support_set_embeddings_div = torch.unsqueeze(
            support_set_embeddings.pow(2).sum(dim=2).sqrt(), 2
        )
        support_set_embeddings_div[support_set_embeddings_div == 0] = 1

        query_embedding = query_embedding / query_embedding_div
        support_set_embeddings = support_set_embeddings / support_set_embeddings_div

    # Compute similarity values
    similarities = query_embedding @ torch.transpose(support_set_embeddings, 1, 2)
    # dim:
    # [batch-size, 1, padding-dim] =
    # [batch-size, 1, emb-dim] x [batch-size, emb-dim, padding-dim]

    # Padding mask
    mask = (
        (~padding_mask.bool()).float().unsqueeze(1)
    ) # remove ~ for old mask

    # Compute similarity values while ignoring padding artefacts
    similarities[torch.isnan(similarities)] = 0.0
    similarities = similarities * mask
    similarity_sums = similarities.sum(
        dim=2
    )  # For every query molecule: Sum over support set molecules

    # Scaling
    if cfg.model.similarityModule.scaling == "1/N":
        stabilizer = torch.tensor(1e-8).float()
        similarity_sums = (
            1 / (2.0 * support_set_size.reshape(-1, 1) + stabilizer) * similarity_sums
        )
    if cfg.model.similarityModule.scaling == "1/sqrt(N)":
        stabilizer = torch.tensor(1e-8).float()
        similarity_sums = (
            1
            / (2.0 * torch.sqrt(support_set_size.reshape(-1, 1).float()) + stabilizer)
            * similarity_sums
        )

    return similarity_sums